In [1]:
import openai
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import client

load_dotenv()


True

In [2]:
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
collection_name = os.getenv("QDRANT_COLLECTION_NAME")


In [3]:
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)


## Download all data From Qdrant

In [4]:
all_points=qdrant_client.scroll(
    collection_name=collection_name,
    limit=400,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [5]:
all_points

([Record(id=1, payload={'description': 'Gvoears Replacement Ear Pads Cushions for Skullcandy Crusher Wireless, Crusher ANC/EVO, Hesh ANC/EVO, Hesh 3 Wireless, Also Fit for Skullcandy Venue Wireless ANC Headphone with Duable Leather Fabric【Perfect Compatibility】These ear pads are made specifically to fit the Skullcandy Hesh 3, Evo, and Crusher series of Headphones. You can comfortably upgrade your ear cushion for skullcandy crusher wireless over-ear headphones, hesh 3 wireless over-ear headphone, crusher evo/anc wireless over-ear headphones, hesh evo/anc wireless over-ear headphones, for skullcandy venue wireless over-ear headphone.【Durable&Comfortable】Crusher wireless replacement pads are designed with soft leather fabric to make your ears do not pain when you wear them for a long time. Durable design make Your headphone can be used for a long time. keep them comfortable and take care of your ears.【Noise Isolation】Earmuffs for skullcandy wireless constructed of premium memory foam, thi

In [7]:
all_points[0][0].payload

{'description': 'Gvoears Replacement Ear Pads Cushions for Skullcandy Crusher Wireless, Crusher ANC/EVO, Hesh ANC/EVO, Hesh 3 Wireless, Also Fit for Skullcandy Venue Wireless ANC Headphone with Duable Leather Fabric【Perfect Compatibility】These ear pads are made specifically to fit the Skullcandy Hesh 3, Evo, and Crusher series of Headphones. You can comfortably upgrade your ear cushion for skullcandy crusher wireless over-ear headphones, hesh 3 wireless over-ear headphone, crusher evo/anc wireless over-ear headphones, hesh evo/anc wireless over-ear headphones, for skullcandy venue wireless over-ear headphone.【Durable&Comfortable】Crusher wireless replacement pads are designed with soft leather fabric to make your ears do not pain when you wear them for a long time. Durable design make Your headphone can be used for a long time. keep them comfortable and take care of your ears.【Noise Isolation】Earmuffs for skullcandy wireless constructed of premium memory foam, this headphone ear covers 

In [8]:
all_context=[{"id":data.payload['parent_asin'],"text":data.payload["description"]} for data in all_points[0] ]

In [9]:
all_context

[{'id': 'B0BTBHYWY4',
  'text': 'Gvoears Replacement Ear Pads Cushions for Skullcandy Crusher Wireless, Crusher ANC/EVO, Hesh ANC/EVO, Hesh 3 Wireless, Also Fit for Skullcandy Venue Wireless ANC Headphone with Duable Leather Fabric【Perfect Compatibility】These ear pads are made specifically to fit the Skullcandy Hesh 3, Evo, and Crusher series of Headphones. You can comfortably upgrade your ear cushion for skullcandy crusher wireless over-ear headphones, hesh 3 wireless over-ear headphone, crusher evo/anc wireless over-ear headphones, hesh evo/anc wireless over-ear headphones, for skullcandy venue wireless over-ear headphone.【Durable&Comfortable】Crusher wireless replacement pads are designed with soft leather fabric to make your ears do not pain when you wear them for a long time. Durable design make Your headphone can be used for a long time. keep them comfortable and take care of your ears.【Noise Isolation】Earmuffs for skullcandy wireless constructed of premium memory foam, this headp

### Render a prompt to generate synthetic Eval Reference dataset

In [ ]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "string",
                "description": "Suggested question",
            },
            "chunck_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the questions",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks",
            },
        },
    },
}
